In [1]:
!nvidia-smi
import torch

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

Tue Jun  9 19:11:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1: Clone Repository and Setup Environment

In [ ]:
import os
from pathlib import Path

# Configuration
REPO_URL = "https://github.com/sattary/ali_proj.git"
BRANCH = "alis_code"  # Change if using different branch
PROJECT_DIR = "ali_proj"

# 1. Clone Repository
if not Path(PROJECT_DIR).exists():
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already cloned. Pulling latest changes...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

%cd {PROJECT_DIR}

# 2. Install uv
print("\nInstalling uv...")
!pip install -q uv

# 3. Set MPLBACKEND for Kaggle compatibility
os.environ['MPLBACKEND'] = 'Agg'
print("\nSet MPLBACKEND=Agg for Kaggle compatibility")

# 4. Sync Dependencies
print("\nSyncing dependencies...")
!uv sync

print("\n✓ Setup complete!")

In [ ]:
# Verify Git Repository
from pathlib import Path
import subprocess

# Check if we're in a git repo
result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
)
if result.returncode == 0:
    repo_root = result.stdout.strip()
    print(f"✓ Git repository found at: {repo_root}")
    print(f"✓ Current directory: {Path.cwd()}")
else:
    print("Error: Not in a git repository!")
    print(f"Current directory: {Path.cwd()}")
    raise RuntimeError("Git repository not found")

# Show repo status
!git status

## Step 2: Generate Training Data

Adjust `num_samples` based on your needs:
- Quick test: 10,000 samples
- Full training: 180,000 samples

In [ ]:
# Generate synthetic interferogram data
!uv run phase-unwrap data generate \
    --num-samples 180000 \
    --shard-size 1000 \
    --out-dir data/kaggle_full \
    --seed 1337

print("\n✓ Data generation complete!")

## Step 3: Hyperparameter Optimization with Optuna

Find optimal hyperparameters before full training. This runs 30 trials with 10 epochs each.

### Configuration:
- `--n-trials 30`: Number of Optuna trials
- `--tune-epochs 10`: Epochs per trial (quick evaluation)
- `--n-workers 2`: Run 2 trials in parallel (one per GPU)
- `--gpu-ids 0,1`: Use GPU 0 and GPU 1

**Speedup**: ~2x faster than single-GPU HPO by running trials in parallel!

**Output**: Best config saved to `runs/optuna/best_config.yaml`. When `--auto-push` is used, results are zipped and pushed to GitHub with data config included.

In [ ]:
# Run Optuna hyperparameter search with parallel trials on 2 GPUs
# This will test different learning rates, batch sizes, and model configurations
# Each GPU runs one trial independently - ~2x speedup!
!uv run phase-unwrap train tune \
    --use-amp \
    --data-dir data/kaggle_full \
    --n-trials 30 \
    --tune-epochs 10 \
    --study-name kaggle_hpo \
    --n-workers 1 \

print("\n✓ Hyperparameter search complete!")
print("\nBest config saved to: runs/optuna/best_config.yaml")

# Display best config
import yaml
from pathlib import Path

config_path = Path("runs/optuna/best_config.yaml")
if config_path.exists():
    with open(config_path, 'r') as f:
        best_config = yaml.safe_load(f)
        print("\n=== Best Hyperparameters ===")
        print(f"Learning Rate: {best_config.get('optim', {}).get('lr', 'N/A')}")
        print(f"Batch Size: {best_config.get('optim', {}).get('batch_size', 'N/A')}")
        print(f"Model Base: {best_config.get('model', {}).get('base', 'N/A')}")
        print(f"EMA Decay: {best_config.get('model', {}).get('ema_decay', 'N/A')}")
        print(f"MAE Weight: {best_config.get('loss', {}).get('w_mae', 'N/A')}")
        print(f"Grad Weight: {best_config.get('loss', {}).get('w_grad', 'N/A')}")
        print(f"Curv Weight: {best_config.get('loss', {}).get('w_curv', 'N/A')}")
else:
    print(f"⚠️  Config file not found at {config_path}")
    print("Using default hyperparameters for training")

## Step 4: Train with Optimal Hyperparameters

Train the model using the **best hyperparameters found by Optuna** with multi-GPU.

### Configuration:
- `--config runs/optuna/best_config.yaml`: Use tuned hyperparameters
- `--epochs 10000`: Train for 10K epochs
- `--multi-gpu`: Use both Kaggle T4 GPUs
- `--batch-size 40`: Total batch size (20 per GPU × 2 GPUs)
- `--run-name exp_10k`: Experiment identifier

**Note**: If `runs/optuna/best_config.yaml` doesn't exist, training will use defaults.

In [ ]:
from pathlib import Path

# Check if best config exists
config_file = "runs/optuna/best_config.yaml"
if Path(config_file).exists():
    print(f"✓ Using optimized config: {config_file}")
    config_arg = f"--config {config_file}"
else:
    print("⚠️  No optimized config found. Using defaults.")
    config_arg = ""

# Full training with multi-GPU, auto-push, and optimized hyperparameters
!uv run phase-unwrap train train \
    --use-amp \
    {config_arg} \
    --epochs 10000 \
    --run-name exp_10k \
    --batch-size 16 \
    --device cuda \
    --data-dir data/kaggle_full \

print("\n✓ Training complete!")

## Step 5: Resume Training (If Interrupted)

If your Kaggle session disconnects, run this cell to resume from the last checkpoint.

**Note**: You can resume on single GPU even if originally trained on multi-GPU (and vice versa). The checkpoint system handles both cases automatically.

In [ ]:
from pathlib import Path

# Check if best config exists
config_file = "runs/optuna/best_config.yaml"
if Path(config_file).exists():
    config_arg = f"--config {config_file}"
else:
    config_arg = ""

# Resume training from checkpoint (with multi-GPU)
!uv run phase-unwrap train train \
    --use-amp \
    {config_arg} \
    --resume runs/exp_10k/final.pth \
    --epochs 10000 \
    --run-name exp_10k \
    --batch-size 16 \
    --device cuda \
    --data-dir data/kaggle_full \

print("\n✓ Training resumed and completed!")

## Step 6: Zip and Download Results

Run this cell to zip the `runs/` directory so you can download it to your local machine for evaluation and visualization.

In [ ]:
import shutil
from IPython.display import FileLink

print("Zipping runs folder...")
shutil.make_archive('training_results', 'zip', 'runs/')
print("✓ Done! You can now download training_results.zip from the file browser.")
display(FileLink('training_results.zip'))

## Tips for Long-Running Training

1. **Hyperparameter Tuning**: The Optuna step (Step 3) runs 30 trials to find optimal hyperparameters. With `--n-workers 2`, trials run in parallel on both GPUs, taking ~15-30 minutes instead of 60-120 minutes on single GPU.

2. **Multi-GPU Training**: Kaggle provides 2x T4 GPUs. The notebook automatically uses both with `--multi-gpu` flag:
   - Batch size is per-GPU: `--batch-size 40` = 20 per GPU × 2 GPUs = 40 total
   - ~1.8x speedup vs single GPU (DataParallel overhead)

3. **Save Notebook**: Kaggle may disconnect. Save your notebook frequently.


5. **Resume Strategy**: If disconnected:
   - Re-run Steps 0-2 (setup)
   - Skip Step 2 (data already generated)
   - Skip Step 3 (HPO already done)
   - Run Step 4 (resume) instead of Step 4
   - Note: You can resume on single GPU even if trained on multi-GPU

6. **Monitor Training**: Check metrics in `runs/exp_10k/metrics.csv`

7. **Storage**: Kaggle provides ~20GB disk. If you run out of space:
   - Delete old runs: `!rm -rf runs/old_run`
   - Clear data: `!rm -rf data/kaggle_full` (after pushing to GitHub)